# 분석 재현 노트북

인사이트 8개, 그림 1·2, R04 기준값(1.30·1.15·1.10) 도출, 반기 재현성 점검, 자동 검증 35항목을 순서대로 실행합니다.

- 원본(`ABP_CONTEST_DATA.csv`)은 공모전 제공 자료라 저장소에 없습니다. `data/`에 두거나 `BC_RAW_CSV`로 경로를 지정하세요.
- 원본 없이 실행만 확인하려면 `python src/make_synthetic_raw.py`로 가짜 원본을 만든 뒤 아래 `RAW`를 바꾸세요.
- 계산은 모두 `src/`의 스크립트 함수를 부르며, 임계값은 `config/rules_thresholds.json`에서 읽습니다.
- 결과에는 실제 수치가 들어가므로 출력이 남은 노트북은 저장소에 올리지 않습니다.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

# 실제 원본: ROOT/'data/ABP_CONTEST_DATA.csv'  |  가짜 원본: ROOT/'data/synthetic/ABP_SYNTHETIC.csv'
RAW = Path(os.environ.get('BC_RAW_CSV', ROOT / 'data' / 'ABP_CONTEST_DATA.csv'))
OUT = ROOT / 'output' / ('synthetic' if 'synthetic' in str(RAW).lower() else 'real')
os.environ['BC_RAW_CSV'] = str(RAW)

import pandas as pd
from common import read_raw, tidy, load_config, load_script
CFG = load_config()
TD = load_script('11_threshold_decision')
INS = load_script('12_insights_figures')
print(RAW)

## 1. 데이터 읽기와 기본 확인

In [ ]:
raw = read_raw(RAW)
t = tidy(raw)
T, idx = TD.base_tables(t)
print(raw.shape, '| 시군구', t.key.nunique(), '| 월', t.ym.nunique(), '| 업종', t.ind.nunique())
print('GENDER_CD', sorted(t.g.unique()), '| AGE_CD', sorted(t.a.unique()))

## 2. 인사이트 1 — 외국인 결제금액과 비중은 다른 방향으로 움직인다

금액이 늘어도 전체 결제가 더 늘면 비중은 줄어듭니다. 성과지표가 금액인지 비중인지 먼저 정해야 한다는 R07 질문의 근거입니다.

In [ ]:
g1, cmp1, s1 = INS.insight1_amount_vs_share(t)
display(cmp1.round(4)); s1

## 3. 인사이트 2 — 대형할인점 결제가 없는 시군구

결제 0은 하위 20%와 따로 '결제 없음'으로 표시합니다(R02). 결제가 잡히지 않은 것을 상점이 없다는 뜻으로 해석하지 않습니다.

In [ ]:
by_sido, s2 = INS.insight2_zero_hypermarket(t)
display(by_sido.to_frame('시군구 수')); s2

## 4. 인사이트 3 — 연령 구간별 편의점·슈퍼마켓 비중

In [ ]:
sh3, s3 = INS.insight3_age_industry(t)
display(sh3[['편의점', '슈퍼마켓', '일반한식']].round(1)); s3

## 5. 인사이트 4 — 외국인 결제 업종 구성 3유형(R03)

외국인 결제 상위 50% 시군구만 대상으로, 편의점+슈퍼마켓 비중 상위 30%는 생활유통형, 그렇지 않고 일반한식 비중 상위 30%는 한식외식형, 나머지는 복합형입니다.

In [ ]:
df4, s4 = INS.insight4_foreign_types(t, T)
s4

## 6. 인사이트 5 — 외국인 결제 비중의 지역 격차(R12)

In [ ]:
w5, s5 = INS.insight5_foreign_share_gap(t, T)
s5

## 7. 인사이트 6 — 월별 변동과 R04 기준값 도출

지역 특이 지수 = (월 일평균 / 6개월 일평균) / 전국 같은 값. 일수와 전국 월 흐름을 뺀 값입니다.

기준값은 규모 구간별 평소 흔들림(견고 표준편차 = MAD × 1.4826)에 `1 + 2.5σ`를 적용해 반올림했습니다.
- 하위 10% 이하 → 1.30, 10~30% → 1.15, 나머지 → 1.10
- '같은 달 상위 20%' 방식은 매달 51곳이 자동으로 걸리고 6개월 중 한 번이라도 걸리는 시군구가 많아 폐기했습니다.

In [ ]:
der, s6 = INS.insight6_season(t, T, idx)
display(der['bands'].round(3))
print('운영값', der['operating'])
print('발동', der['flagged_regions'], f"({der['flagged_share']:.0%})", {g: v['flagged'] for g, v in der['by_group'].items()})
print('월별', der['flagged_by_month'])
print('폐기한 대안(같은 달 상위 20%) 한 번 이상 걸리는 비율', f"{s6['rejected_same_month_top20_any_share']:.0%}")

## 8. 인사이트 7 — 성별·연령 미상 비중의 지역 차이

In [ ]:
u7, s7 = INS.insight7_unknown(t, T)
s7

## 9. 인사이트 8 — 군집 대신 유사 지자체(최근접 이웃)

업종 9 + 연령 5 + 외국인 비중 + log 결제규모를 표준화해 KMeans 실루엣을 확인했고, 분리도가 낮아 최근접 이웃 5곳을 유사 지자체로 씁니다.

In [ ]:
s8 = INS.insight8_cluster(t, T)
s8

## 10. 반기 재현성 점검 (1~3월 vs 4~6월)

같은 규칙을 두 반기에 따로 적용했을 때 같은 지역이 뽑히는 정도(Jaccard, R03은 유형 일치율)입니다. 0.8 안팎 이상이면 안정적으로 봅니다.

In [ ]:
rep = TD.reproducibility_summary(t, T)
pd.DataFrame({'재계산': rep, '보고값': CFG['reproducibility']['reported']}).round(2)

In [ ]:
# 하위 비율을 바꿔 가며 본 재현성(임계값 후보 비교)
display(pd.DataFrame(TD.r02_reproducibility(t, T)).T.round(2))
display(pd.DataFrame(TD.r12_reproducibility(t, T)).T.round(2))
display(pd.DataFrame({f'{k[0]}/{k[1]}': {'agreement': v['agreement'], **v['distribution']} for k, v in TD.r03_reproducibility(t, T).items()}).T)

## 11. 그림 1·2

In [ ]:
res = INS.run_all(RAW, OUT, figures=True)
from IPython.display import Image
display(Image(str(OUT / 'figure1.png'))); display(Image(str(OUT / 'figure2.png')))

## 12. 근거 파일 생성과 자동 검증 35항목

실제 원본이면 `09 real`(외부 자료 필요)과 `--mode real`, 가짜 원본이면 `--no-external`과 `--mode check`로 돌립니다.

In [ ]:
synthetic = 'synthetic' in str(RAW).lower()
real_json = ROOT / 'private' / ('smoke_real.json' if synthetic else 'review_evidence_real_v2.1.json')
py = sys.executable
subprocess.run([py, str(ROOT/'src/09_make_evidence.py'), 'synthetic'], check=True)
subprocess.run([py, str(ROOT/'src/09_make_evidence.py'), 'real', '--raw', str(RAW), '--out', str(real_json)] + (['--no-external'] if synthetic else []), check=True)
r = subprocess.run([py, str(ROOT/'src/10_verify_evidence.py'), '--raw', str(RAW), '--real', str(real_json),
                    '--mode', 'check' if synthetic else 'real', '--report', str(OUT/'validation_35.json')], capture_output=True, text=True)
print(r.stdout[-1500:])